# Windowing & Handling Class Imbalance

Strategy: 20 timesteps, step 5, label = last timestep. Mild oversampling on Train.

In [1]:
import pandas as pd
import numpy as np
import os

DATA_DIR = '../data'

In [2]:
df_train = pd.read_csv(f'{DATA_DIR}/train_feat.csv')
df_test = pd.read_csv(f'{DATA_DIR}/test_feat.csv')

FEATURES = ['Pressure (bar)', 'Flow Rate (L/s)', 'Temperature (°C)',
            'pressure_delta', 'flow_zscore', 'pf_ratio']

WINDOW_SIZE = 20
STEP_SIZE = 5

## 1. Sliding Windows

In [3]:
def create_windows(df):
    X_list, y_list = [], []
    for s_id, group in df.groupby('Sensor_ID'):
        vals = group[FEATURES].values
        labels = group['Leak Status'].values

        for start in range(0, len(group) - WINDOW_SIZE + 1, STEP_SIZE):
            end = start + WINDOW_SIZE
            X_list.append(vals[start:end])
            # Baseline-style label: leak if ANY timestep in the window is leak
            y_list.append(int(labels[start:end].max() > 0))

    return np.array(X_list), np.array(y_list)

X_train, y_train = create_windows(df_train)
X_test, y_test = create_windows(df_test)

print('Train:', X_train.shape, y_train.shape, f'| Leaks: {y_train.sum()} ({y_train.mean()*100:.1f}%)')
print('Test:', X_test.shape, y_test.shape, f'| Leaks: {y_test.sum()} ({y_test.mean()*100:.1f}%)')

Train: (859, 20, 6) (859,) | Leaks: 320 (37.3%)
Test: (191, 20, 6) (191,) | Leaks: 69 (36.1%)


## 2. Mild Oversampling (Train Only)
Focus on leaks, avoiding extreme overfitting by maintaining some imbalance.

In [4]:
leak_idx = np.where(y_train == 1)[0]

# Duplicate leaks 2x extra for mild augmentation
X_leaks_extra = np.repeat(X_train[leak_idx], 2, axis=0)
y_leaks_extra = np.repeat(y_train[leak_idx], 2, axis=0)

X_train_bal = np.concatenate([X_train, X_leaks_extra], axis=0)
y_train_bal = np.concatenate([y_train, y_leaks_extra], axis=0)

# Shuffle
rng = np.random.default_rng(42)
shuf = rng.permutation(len(X_train_bal))
X_train_bal = X_train_bal[shuf]
y_train_bal = y_train_bal[shuf]

print('After Mild Oversampling:')
print('Train:', X_train_bal.shape, y_train_bal.shape, f'| Leaks: {y_train_bal.sum()} ({y_train_bal.mean()*100:.1f}%)')

After Mild Oversampling:
Train: (1499, 20, 6) (1499,) | Leaks: 960 (64.0%)


## 3. Save Arrays

In [5]:
np.save(f'{DATA_DIR}/X_train.npy', X_train_bal)
np.save(f'{DATA_DIR}/y_train.npy', y_train_bal)
np.save(f'{DATA_DIR}/X_test.npy', X_test)
np.save(f'{DATA_DIR}/y_test.npy', y_test)
print('Arrays successfully saved (train/test only).')

Arrays successfully saved (train/test only).
